# Speech Text Emotion Recognition

## Text To Features

In [ ]:
# Basic imports
import os
import sys
import glob
import pickle
import warnings
import re
from dotenv import load_dotenv

# Data manipulation and numerical processing libraries
import numpy as np
import pandas as pd

# Audio processing libraries
import librosa
import librosa.display
import soundfile as sf
import opensmile

# Visualization library
import matplotlib.pyplot as plt

# Machine learning and evaluation tools
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Progress bar for loops
from tqdm.notebook import tqdm

# Ignore warnings for cleaner output
warnings.filterwarnings("ignore")

# Custom feature extraction modules from psychai (assuming psychai is installed in your environment)
import psychai.feature.feature_extraction.feature_retriever
import psychai.feature.feature_extraction.feature_processor
import psychai.speech_acoustics.feature_extraction.feature_extraction

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

import jieba
import string
import pandas as pd
import os, sys
import psychai.data_visualization.chart
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize feature processor from psychai
feature_processor = psychai.feature.feature_extraction.feature_processor.FeatureProcessor()

# Define the current directory path. This is useful if you have a custom module or scripts located in a current
# directory that you want to import into this script.
current_dir = os.path.abspath('../')

# Add the parent directory path to the Python path (sys.path), which allows importing modules from it.
sys.path.append(current_dir)

import local_utilities
import local_data_loader
from datetime import datetime
# Set base directories and file paths

load_dotenv(override=True)
huggingface_cache_location = os.getenv("huggingface_cache_location")
datasets_cache_location = os.getenv("datasets_cache_location")
resutls_location = os.getenv("results_location")
resource_path= os.getenv("resources_location")

today = datetime.now().strftime('%Y%m%d')

csv_path = os.path.join(resutls_location, "examples", "paper5_mmer_moral","speech_text","csv")
record_path =  os.path.join(csv_path,"result_20241107.csv")
save_path_speech_text =  os.path.join(csv_path,f"result_speech_text_{today}.csv")
save_path_speech_sentiment =  os.path.join(csv_path,f"result_speech_sentiment_{today}.csv")
save_path_speech_sentiment_concatenated =  os.path.join(csv_path,f"result_speech_sentiment_concatenated_{today}.csv")

data_helper = local_utilities.DataHelper()


In [ ]:
def preprocess_text(text):
    # Tokenize the Chinese text using jieba
    tokens = jieba.lcut(text)
    
    # Remove punctuation
    tokens = [word for word in tokens if word not in string.punctuation]
    
    return ' '.join(tokens)


In [ ]:
df_read = pd.read_csv(record_path)
df_read["Processed_Text"] = df_read["text"].apply(data_helper.clean_llm_returned_text)
df_read= df_read.reset_index(drop=True)
df_read

In [ ]:
# Function to remove all non-Chinese characters
def remove_non_chinese(text):
    return re.sub(r'[^\u4e00-\u9fff]', '', text)

# Apply the function to the 'text' column
df_read['Processed_Text'] = df_read['Processed_Text'].apply(remove_non_chinese)

# Define a list of specific Chinese substrings you want to remove from the 'text' column
substrings_to_remove = [
    "看完短片之后你是否感到这世上的人们实在是很好", 
    "看完这篇短你是否想要成为一个更好的人具体描述", 
    "问健一让我回答我给我的最强烈的印象和情绪感受", 
    "口头影片问卷二,我有一些感到一些生理反应",
]

# Function to remove specified substrings from a text
def remove_substrings(text, substrings):
    for substring in substrings:
        text = text.replace(substring, "")  # Replace each substring with an empty string
    return text.strip()  # Strip any leading/trailing whitespace

# Apply the removal function to each row in the 'text' column
df_read['Processed_Text'] = df_read['Processed_Text'].apply(lambda x: remove_substrings(x, substrings_to_remove))

# Apply preprocessing to the text data
df_read['Processed_Text'] = df_read['Processed_Text'].apply(preprocess_text)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer() 

# Convert the processed text data to TF-IDF features
fitted_tfidf = tfidf.fit_transform(df_read['Processed_Text'])

# Step 4: Convert the result to a DataFrame for easy viewing
tfidf_df = pd.DataFrame(fitted_tfidf.toarray(), columns=tfidf.get_feature_names_out())

tfidf_df

##### Show only the first row with non-zero entries

In [ ]:
# Select only the first row of tfidf_df
first_row_df = tfidf_df.iloc[[1]]  # Use double brackets to keep it as a DataFrame

# Remove columns where the value is zero in the first row
first_row_df_non_zero = first_row_df.loc[:, (first_row_df != 0).any(axis=0)]

# Display the resulting DataFrame
print(first_row_df_non_zero)

In [ ]:
# Step 4: Concatenate the TF-IDF DataFrame with the original df_read
df_read_with_tfidf = pd.concat([df_read.reset_index(drop=True), tfidf_df], axis=1)

In [ ]:
# Convert the existing selection to a list and add the new column at the beginning
selected_columns = df_read_with_tfidf.columns[16:]
selected_columns = selected_columns.insert(0, "attribute_value")
selected_columns = selected_columns.insert(0, "user_id")
selected_columns = selected_columns.insert(0, "Group")
# Reorder df_filtered_speech_acoustics with the new column order
# Ensure the new column exists in the DataFrame; otherwise, this will raise an error
df_trimmed_speech_text = df_read_with_tfidf[selected_columns]
df_trimmed_speech_text.to_csv(save_path_speech_text)
df_trimmed_speech_text

## Text To Sentiment

In [ ]:
model_name = "SchuylerH/bert-multilingual-go-emtions"

# Load the model and tokenizer
tokenizer_go_emotions = AutoTokenizer.from_pretrained(model_name,cache_dir=huggingface_cache_location)
model_go_emotions = AutoModelForSequenceClassification.from_pretrained(model_name,cache_dir=huggingface_cache_location)

In [ ]:
classifier_go_emotions  = pipeline("sentiment-analysis", model = model_go_emotions, 
                                   tokenizer = tokenizer_go_emotions, device=device,
                                   return_all_scores=True  # This returns scores for all emotions
                                   )

In [ ]:
df_read_sentiment = df_read.copy()

# Process the entire 'text' column in a single batch
results = classifier_go_emotions(list(df_read_sentiment['Processed_Text'])[0])
results

In [ ]:
import pandas as pd

df_read_sentiment = df_read.copy()

# Process the entire 'Processed_Text' column in a single batch
results = classifier_go_emotions(list(df_read_sentiment['Processed_Text']), return_all_scores=True)

# Extract unique emotion labels (from the first result assuming consistent labels across batches)
emotion_labels = [emotion['label'] for emotion in results[0]]

# Initialize columns for each emotion
for label in emotion_labels:
    df_read_sentiment[label] = 0.0

# Populate the DataFrame with scores for each emotion
for i, result in enumerate(results):
    for emotion in result:
        df_read_sentiment.at[i, emotion['label']] = emotion['score']

# Function to determine the dominant emotion based on conditions
def relabel_emotion(row):
    word_count = len(row['Processed_Text'].split())
    max_label = row[emotion_labels].idxmax()  # Find the emotion with the highest score
    max_score = row[max_label]
    if max_score < 0.6 or word_count < 10:
        return "na"
    return max_label

# Apply relabeling to the DataFrame
df_read_sentiment['emotion_relabeled'] = df_read_sentiment.apply(relabel_emotion, axis=1)


# Display the updated DataFrame
print(df_read_sentiment)


In [ ]:
df_read_sentiment.to_csv(save_path_speech_sentiment)
df_read_sentiment

In [ ]:
# Step 4: Concatenate the TF-IDF DataFrame with the original df_read
df_merged_speech_sentiment= pd.concat([df_trimmed_speech_text.reset_index(drop=True), df_read_sentiment.iloc[:,16:-1]], axis=1)
df_merged_speech_sentiment.to_csv(save_path_speech_sentiment_concatenated)
# Display the resulting DataFrame
df_merged_speech_sentiment


## Footnote
- Copyright (c) 2026 Ivan Liu. Licensed under CC BY 4.0.
- Last Update: 2024
- Env: psychai241104, env41124
- References: 
    - None